In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as sns

## Predicting Trotro/Taxi Fares in KumasiThis notebook builds a regression model that estimates the fare (in GHS) of a shared taxi / trotro trip within the Kumasi metropolis, based on trip characteristics such as distance, traffic conditions, weather, market days (e.g. Kejetia/Adum market days), and vehicle type.

**Read the data**

In [ ]:
df = pd.read_csv('Kumasi_Trotro_Fares.csv')

**Display some rows**

In [ ]:
df.head()

**View the number of rows and columns in the data**

In [ ]:
df.shape  # getting the count of rows and columns in the dataframe

**View summary statistics**

In [ ]:
df.describe()

**View data types and non-null counts**

In [ ]:
df.info()

**Look for duplicate rows in the dataset**

In [ ]:
df.loc[df.duplicated()]

No duplicate rows.

**Get the name of the columns**

In [ ]:
df.columns

**Check for null values**

In [ ]:
df.isna().sum()

No missing values.

## Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(20,8))plt.subplot(1,2,1)plt.title('Trotro/Taxi Fare Distribution (GHS)')sns.histplot(df.fare, kde=True)plt.subplot(1,2,2)sns.boxplot(x=df.fare)plt.title('Fare Spread')plt.show()

In [ ]:
print(df.fare.describe(percentiles=[0.25,0.50,0.75,0.85,0.90,1]))

Inference:The fare distribution is mildly right-skewed, with most trips within the Kumasi metro costing between GHS 15 and GHS 45, and a smaller number of longer/private trips (e.g. Kumasi Airport to Suame, or Asokwa to Bantama in heavy rain) pushing the fare higher.

**Visualising Categorical Data**

In [ ]:
categorical_list = [x for x in df.columns if df[x].dtype == 'object']print(categorical_list)

In [ ]:
plt.figure(figsize=(15, 6))plt.subplot(1,3,1)plt1 = df['major_road'].value_counts().plot(kind='bar')plt.title('Major Road Histogram')plt1.set(xlabel='major_road', ylabel='Frequency')plt.subplot(1,3,2)plt1 = df['peak_hour'].value_counts().plot(kind='bar')plt.title('Peak Hour Histogram')plt1.set(xlabel='peak_hour', ylabel='Frequency')plt.subplot(1,3,3)plt1 = df['rainy'].value_counts().plot(kind='bar')plt.title('Rainy Day Histogram')plt1.set(xlabel='rainy', ylabel='Frequency')plt.tight_layout()plt.show()

In [ ]:
plt.figure(figsize=(15, 5))plt.subplot(1,3,1)plt1 = df['market_day'].value_counts().plot(kind='bar', color='r')plt.title('Market Day Histogram')plt1.set(xlabel='market_day', ylabel='Frequency')plt.subplot(1,3,2)plt1 = df['ac_vehicle'].value_counts().plot(kind='bar', color='g')plt.title('AC Vehicle Histogram')plt1.set(xlabel='ac_vehicle', ylabel='Frequency')plt.subplot(1,3,3)plt1 = df['vehicle_type'].value_counts().plot(kind='bar', color='orange')plt.title('Vehicle Type Histogram')plt1.set(xlabel='vehicle_type', ylabel='Frequency')plt.tight_layout()plt.show()

In [ ]:
plt.figure(figsize=(10,5))plt.subplot(1,2,1)plt.title('Peak Hour vs Fare')sns.boxplot(x=df.peak_hour, y=df.fare, palette='cubehelix')plt.subplot(1,2,2)plt.title('Rainy Day vs Fare')sns.boxplot(x=df.rainy, y=df.fare, palette='PuBuGn')plt.show()

Inference: Boarding during peak hour and on a rainy day both push the fare upward, which matches everyday experience of hailing a taxi in Adum or Kejetia during rush hour or a downpour.

In [ ]:
plt.figure(figsize=(10,5))plt.subplot(1,2,1)plt.title('Market Day vs Fare')sns.boxplot(x=df.market_day, y=df.fare, palette='cubehelix')plt.subplot(1,2,2)plt.title('AC Vehicle vs Fare')sns.boxplot(x=df.ac_vehicle, y=df.fare, palette='PuBuGn')plt.show()

Inference: Trips on a big market day (Kejetia/Adum) and trips in an air-conditioned vehicle tend to cost more.

In [ ]:
plt.figure(figsize=(8,5))sns.boxplot(x=df.vehicle_type, y=df.fare, palette='Set2')plt.title('Vehicle Type vs Fare')plt.show()

Inference: Shared trotro trips are the cheapest, chartered (dropping) trips cost more, and private hires are the most expensive, as expected.

**Visualising numerical data**

In [ ]:
numerical_list = [x for x in df.columns if df[x].dtype in ('int64','float64')]print(numerical_list)

In [ ]:
def scatter(x, fig):    plt.subplot(3,2,fig)    plt.scatter(df[x], df['fare'])    plt.title(x + ' vs Fare')    plt.ylabel('Fare')    plt.xlabel(x)plt.figure(figsize=(10,14))scatter('distance_km', 1)scatter('trip_duration_min', 2)scatter('num_stops', 3)scatter('traffic_level', 4)scatter('luggage_items', 5)plt.tight_layout()plt.show()

Inference: distance_km and trip_duration_min show the strongest positive correlation with fare.

In [ ]:
sns.pairplot(df[numerical_list + ['fare']])plt.show()

In [ ]:
cor_matrix = df[numerical_list + ['fare']].corr()plt.figure(figsize=(10, 7))sns.heatmap(cor_matrix, annot=True, cmap='coolwarm', linewidths=0.5)plt.title('Correlation Heatmap')plt.show()

## Preparing the data for modelling

Dummy Variables

In [ ]:
# Defining the map functiondef dummies(x, df):    temp = pd.get_dummies(df[x], drop_first=True).astype(int)    df = pd.concat([df, temp], axis=1)    df.drop([x], axis=1, inplace=True)    return df# Applying the function to yes/no columnsfor col in ['major_road','peak_hour','rainy','market_day','ac_vehicle','safe_route']:    df = dummies(col, df)# vehicle_type has 3 levels, so it needs its own dummy columnsvtype_dummies = pd.get_dummies(df['vehicle_type'], prefix='vtype', drop_first=True).astype(int)df = pd.concat([df, vtype_dummies], axis=1)df.drop(['vehicle_type'], axis=1, inplace=True)

In [ ]:
df.head()

In [ ]:
df.shape

**Train / test split**

In [ ]:
from sklearn.model_selection import train_test_splitnp.random.seed(0)df_train, df_test = train_test_split(df, train_size=0.75, test_size=0.25, random_state=100)

**Feature scaling**

In [ ]:
from sklearn.preprocessing import MinMaxScalerscaler = MinMaxScaler()df_train[numerical_list] = scaler.fit_transform(df_train[numerical_list])

In [ ]:
df_train.head()

In [ ]:
# Dividing data into X and y variablesy_train = df_train.pop('fare')X_train = df_train

## Feature selection with RFE

In [ ]:
from sklearn.feature_selection import RFEfrom sklearn.linear_model import LinearRegressionimport statsmodels.api as smfrom statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
rfe = RFE(estimator=LinearRegression(), n_features_to_select=9)rfe = rfe.fit(X_train, y_train)

In [ ]:
list(zip(X_train.columns, rfe.support_, rfe.ranking_))

In [ ]:
X_train.columns[rfe.support_]

In [ ]:
X_train_rfe = X_train[X_train.columns[rfe.support_]]X_train_rfe.head()

In [ ]:
def build_model(X, y):    X = sm.add_constant(X)  # Adding the constant for intercept    lm = sm.OLS(y, X).fit()  # fitting the model    print(lm.summary())  # model summary    return Xdef checkVIF(X):    vif = pd.DataFrame()    vif['Features'] = X.columns    vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]    vif['VIF'] = round(vif['VIF'], 2)    vif = vif.sort_values(by='VIF', ascending=False)    return vif

MODEL 1

In [ ]:
X_train_new = build_model(X_train_rfe, y_train)

Dropping the feature with the highest p-value greater than the significance level of 0.05.

In [ ]:
# Inspect p-values from the summary above and drop the weakest predictor, e.g.:# X_train_new = X_train_new.drop(["num_stops"], axis=1)X_train_new = X_train_new.drop(["num_stops"], axis=1)

Model 2

In [ ]:
X_train_new = build_model(X_train_new, y_train)

In [ ]:
checkVIF(X_train_new)

Dropping any feature with a very high VIF score to reduce multicollinearity.

In [ ]:
X_train_new = X_train_new.drop(["trip_duration_min"], axis=1)

Model 3

In [ ]:
X_train_new = build_model(X_train_new, y_train)

In [ ]:
checkVIF(X_train_new)

Model is ready — let's verify it through residual analysis.

In [ ]:
lm = sm.OLS(y_train, X_train_new).fit()y_train_fare = lm.predict(X_train_new)

In [ ]:
# Plot the histogram of the error termsfig = plt.figure()sns.histplot((y_train - y_train_fare), kde=True, bins=20)fig.suptitle('Error Terms', fontsize=20)plt.xlabel('Errors', fontsize=18)plt.show()

Error terms approximately follow a normal distribution centred at zero, so the linear regression assumption of normally distributed residuals is reasonably satisfied.

**Predicting on the Test set**

In [ ]:
# scale the test set numerical valuesdf_test[numerical_list] = scaler.transform(df_test[numerical_list])

In [ ]:
# Dividing into X and yy_test = df_test.pop('fare')X_test = df_test

In [ ]:
# selecting the chosen features from the train set to the test setX_train_new = X_train_new.drop('const', axis=1)X_test_new = X_test[X_train_new.columns]# Adding a constant variableX_test_new = sm.add_constant(X_test_new)

In [ ]:
# Making predictionsy_pred = lm.predict(X_test_new)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_errorprint('R-squared on test set:', r2_score(y_test, y_pred))print('MAE on test set (GHS):', mean_absolute_error(y_test, y_pred))

In [ ]:
# EVALUATION OF THE MODELfig = plt.figure()plt.scatter(y_test, y_pred)fig.suptitle('y_test vs y_pred', fontsize=20)plt.xlabel('y_test (actual fare, GHS)', fontsize=14)plt.ylabel('y_pred (predicted fare, GHS)', fontsize=14)plt.show()

In [ ]:
print(lm.summary())

## ConclusionThe final model uses trip distance, traffic level, luggage, and trip conditions (peak hour, rain, market day, air-conditioning, main road, safe route) and vehicle type to estimate fares for shared taxi/trotro trips in the Kumasi metropolis. The R-squared and the coefficient p-values in the summary above show how well the model explains fare variation, and can help commuters and drivers in Kumasi anticipate a fair trip price under different conditions.